In [1]:
import numpy as np
from tensorflow.keras.models import Sequential

# RNN layer
from tensorflow.keras.layers import SimpleRNN

# Fully connected output layer
from tensorflow.keras.layers import Dense

# Converts word IDs into dense vectors
from tensorflow.keras.layers import Embedding

# Converts text into sequences of numbers
from tensorflow.keras.preprocessing.text import Tokenizer

# Makes all sequences the same length
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
texts = [

    "I love this movie",
    "This film is amazing",
    "Very good acting",
    "Excellent story",
    "I hate this movie",
    "Terrible film",
    "Very boring story",
    "Worst acting ever"

]

# Labels
# 1 = Positive
# 0 = Negative

labels = np.array([
    1, 1, 1, 1,
    0, 0, 0, 0
])

In [7]:
# Create tokenizer object
tokenizer = Tokenizer()

# Learn all unique words from the dataset and assign an integer ID to each word
tokenizer.fit_on_texts(texts)

# Create a dictionary:
# word -> integer ID
print("Word Index:")
print(tokenizer.word_index)

# Convert each sentence into a sequence of numbers based on the word index
sequences = tokenizer.texts_to_sequences(texts)
print("\nSequences:")
print(sequences)

# Example:

# "I love this movie"
# may become
# [3, 4, 1, 2]



Word Index:
{'this': 1, 'i': 2, 'movie': 3, 'film': 4, 'very': 5, 'acting': 6, 'story': 7, 'love': 8, 'is': 9, 'amazing': 10, 'good': 11, 'excellent': 12, 'hate': 13, 'terrible': 14, 'boring': 15, 'worst': 16, 'ever': 17}

Sequences:
[[2, 8, 1, 3], [1, 4, 9, 10], [5, 11, 6], [12, 7], [2, 13, 1, 3], [14, 4], [5, 15, 7], [16, 6, 17]]


In [11]:
X = pad_sequences(
    sequences,
    maxlen=4, # Maximum length of sequences. Shorter sequences will be padded, longer ones will be truncated
)

print("\nPadded Sequences:")
print(X)

model = Sequential()


model.add(
    Embedding(

        # Vocabulary size
        input_dim=len(tokenizer.word_index) + 1,

        # Each word becomes a vector

        # with 8 dimensions

        output_dim=8,

        # Input sentence length

        input_length=4

    )

)

# Example:

#

# Word ID: 5

#

# becomes

#

# [0.24, -0.17, 0.88, ...]

#

# instead of a single integer.

# ------------------------------------------

# RNN Layer

# ------------------------------------------

model.add(

    SimpleRNN(

        # Number of memory units

        8

    )

)

# The RNN reads words one by one

#

# Word 1 --> Hidden State

# Word 2 --> Hidden State

# Word 3 --> Hidden State

# Word 4 --> Hidden State

#

# Previous information is carried forward

# through the hidden state.

# ------------------------------------------

# Output Layer

# ------------------------------------------

model.add(

    Dense(

        # One neuron because

        # we predict Positive/Negative

        1,

        activation='sigmoid'

    )

)

# Sigmoid output:

#

# 0.00 -> Negative

# 1.00 -> Positive

#

# Example:

#

# 0.92 = 92% Positive

# ==========================================

# STEP 5: Compile model

# ==========================================

model.compile(

    # Optimizer updates weights

    optimizer='adam',

    # Loss function for binary classification

    loss='binary_crossentropy',

    # Accuracy metric

    metrics=['accuracy']

)

# Show architecture

model.summary()

# ==========================================

# STEP 6: Train model

# ==========================================

model.fit(

    X,

    labels,

    # Number of complete passes

    # through training data

    epochs=200,

    # Suppress training output

    verbose=0

)

# ==========================================

# STEP 7: Test with new review

# ==========================================

new_text = [

    "Amazing acting and good story"

]

# Convert sentence into word IDs

new_sequence = tokenizer.texts_to_sequences(

    new_text

)

print("\nNew Sequence:")

print(new_sequence)

# Pad to length 4

new_padded = pad_sequences(

    new_sequence,

    maxlen=4

)

print("\nPadded Sequence:")

print(new_padded)

# Predict sentiment

prediction = model.predict(

    new_padded

)

print("\nPrediction Probability:")

print(prediction[0][0])

# Convert probability into class

if prediction[0][0] > 0.5:

    print("Sentiment: Positive 😊")

else:

    print("Sentiment: Negative ☹️")


Padded Sequences:
[[ 2  8  1  3]
 [ 1  4  9 10]
 [ 0  5 11  6]
 [ 0  0 12  7]
 [ 2 13  1  3]
 [ 0  0 14  4]
 [ 0  5 15  7]
 [ 0 16  6 17]]


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_5 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


New Sequence:
[[10, 6, 11, 7]]

Padded Sequence:
[[10  6 11  7]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step

Prediction Probability:
0.8574085
Sentiment: Positive 😊
